In [ ]:
print('Ritu')

In [1]:
from langchain_ai21.chat_models import ChatAI21
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage
from langgraph.types import interrupt,Command 
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from typing import TypedDict,Annotated, List, Dict, Literal
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv
import json
import sqlite3
import os


load_dotenv()

model = ChatAI21(model = 'jamba-mini-1.7-2025-07')
searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)


C:\Users\kaushal\AppData\Local\Temp\ipykernel_25848\2853768797.py:20: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  searchTool = TavilySearchResults(max_results=3)


In [ ]:
# model = ChatAI21(
#     model="jamba-mini-1.7-2025-07",
#     # api_key="cc97c84e-8137-476c-8324-ce151112c72d",
#     # base_url="https://api.ai21.com/studio/v1"
# )
model.invoke('Hi')

# Simple Flow

In [ ]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 

In [ ]:
DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def except_detail_prompt(detail_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{detail_text}

Please convert this into valid JSON with the following structure:
{{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {{
            "key_point":
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt

In [ ]:
def is_valid_brackets(s: str) -> bool:
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack:
         s += bracket_map[char]
     return s

In [ ]:
def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text


In [ ]:
class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    num_slides: int

In [ ]:
def chat_node(state: PptState):
    """Main chat node that processess messages"""
    messages = state["messages"]
    result = model_with_tools.invoke(messages)
    return {'messages':result}

In [ ]:
def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state["topic"]
    num_slides = state['num_slides']

    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )

    messages = [OUTLINE_SYSTEM_PORMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('generate_outline_node result',type(result.content))
    try:
        outline = json_to_python(result.content)
        outline = json.loads(outline)
    except json.JSONDecodeError:
        
        outline = json.loads(is_valid_brackets(outline))
        # fix_prompt = except_outline_prompt(result.content)
        # try:
        #     fix_result = model.invoke([fix_prompt])
        #     fixed_text = fix_result.content.strip()
        #     outline = json_to_python(fixed_text)
        #     outline = json.loads(outline)
        # except json.JSONDecodeError:
        #     outline = {
        #         'title': topic,
        #         'total_slide': num_slides,
        #         'slides': []
        #     }
    print('outline',outline['slides'])

    return {
        'messages':[result],
        'outline':outline,
        'current_slide_index':0
            }

In [ ]:
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    if current_index > state['num_slides']:
        return
    
    # print('outline',type(outline),outline)
    print('current_index',current_index)
    current_slide = outline['slides'][int(current_index)]
    # print('current_slide',current_slide)
    # print('slide_title',current_slide['slide_title'])
    # print('key_points',current_slide['key_points'])
    # print('content_type',current_slide['content_type'])

    prompt = HumanMessage(
        content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        print('before is_valid_brackets',detailed_slide)
        print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except:
            fix_prompt = except_detail_prompt(detailed_slide)
            try:
                fix_result = model.invoke([fix_prompt])
                fixed_text = fix_result.content.strip()
                detailed_slide = json_to_python(fixed_text)
                detailed_slide = json.loads(is_valid_brackets(detailed_slide))
            except json.JSONDecodeError as e:
                print('last erroe',e)
                print('detailed_slide',detailed_slide)
                detailed_slide = {
                    'slide_number': current_slide['slide_number'],
                    'slide_title': current_slide['slide_title'],
                    'detailed_content':[],
                    'row_response': detailed_slide
                }
    
    detailed_slides = state.get('detailed_slides',[])
    detailed_slides.append(detailed_slide)
    return {
        'messages':[result],
        'detailed_slides':detailed_slides,
        'current_slide_index': current_index +1
    }

In [ ]:
def should_coutine_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index',0)
    total_slides = len(state.get('outline',{}).get('slides',[]))

    if current_index < total_slides:
        return "continue"
    else:
        return "end"

In [ ]:
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node("generate_slide_detail",generate_slide_detail_node)
workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
workflow.add_edge("generate_outline","generate_slide_detail")
workflow.add_conditional_edges(
    "generate_slide_detail",should_coutine_slides,
    {
        "continue":"generate_slide_detail",
        "end":END
    }

)

conn = sqlite3.connect("graph.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

ppt_generator = workflow.compile(checkpointer = checkpointer)

In [ ]:
initial_state = {
        'messages': [],
        'outline': {},
        'detailed_slides': [],
        'current_slide_index': 0,
        'topic': 'what is recursion in python',
        'num_slides': 1
    }
config = {"configurable":{"thread_id":"user-123"}}
result = ppt_generator.invoke(initial_state,config=config)

In [ ]:
result

# HITL

In [5]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 


DETAIL_SYSTEM_PROMPT = SystemMessage(
    content="""
You are an expert content writer for presentations.

Your task: Generate DETAILED, ENGAGING content for a specific slide.

For each key point:
- Provide 2-3 sentences of explanation
- Include relevant examples, statistics, or facts
- Make it clear, concise, and presentation-ready
- Use simple language that's easy to understand

Output format:
{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {
            "key_point":
        }
    ]
}

"""
)

def except_detail_prompt(detail_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{detail_text}

Please convert this into valid JSON with the following structure:
{{
    "slide_number": number,
    "slide_title": "Title",
    "detailed_content": [
        {{
            "key_point":
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt

def is_valid_brackets(s: str) -> bool:
     if s == '':
          return
     stack = []
     bracket_map = {
         '(' : ')',
         '{' : '}',
         '[' : ']',
     }

     if s[0] != '{':
         s = '{' +s

     for char in s:
        if char in bracket_map:
            stack.append(char)

        elif char in bracket_map.values():
                stack.pop()

     if s[-1] not in ['"',']','}']:
         s = s+'"'
     for char in stack:
         s += bracket_map[char]
     return s

def json_to_python(fixed_text):
    if "```json" in fixed_text:
        fixed_text = fixed_text.replace("```json", "").replace("```", "").strip()
    elif "```" in fixed_text:
        fixed_text = fixed_text.replace("```", "").strip()
    return fixed_text

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    feedback: str
    action: str
    num_slides: int

def chat_node(state: PptState):
    """Main chat node that processess messages"""
    messages = state["messages"]
    result = model_with_tools.invoke(messages)
    return {'messages':result}


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    topic = state["topic"]
    num_slides = state['num_slides']

    prompt = HumanMessage(
        content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    )

    messages = [OUTLINE_SYSTEM_PORMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('generate_outline_node result',type(result.content))
    try:
        outline = json_to_python(result.content)
        outline = json.loads(outline)
    except json.JSONDecodeError:
        
        outline = json.loads(is_valid_brackets(outline))

    return {
        'messages':[result],
        'outline':outline,
        'current_slide_index':0
            }

def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    if current_index > state['num_slides']:
        return
    
    print('current_index',current_index)
    current_slide = outline['slides'][int(current_index)]

    prompt = HumanMessage(
        content=f"""Generate detailede content for this slide:
Slide Number: {current_slide['slide_title']}
Key Points: {', '.join(current_slide['key_points'])}
Content Type: {current_slide['content_type']}

Provide comprehensive, presentation-ready content."""
    )

    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    # print('result.content',result.content)

    try:
        # print('inside first try')
        detailed_slide = json_to_python(result.content)
        detailed_slide = json.loads(detailed_slide)
        
    except json.JSONDecodeError as e:
        print('before is_valid_brackets',detailed_slide)
        print('after is_valid_brackets',is_valid_brackets(detailed_slide))
        try:
            detailed_slide = json.loads(is_valid_brackets(detailed_slide))
        except:
            fix_prompt = except_detail_prompt(detailed_slide)
            try:
                fix_result = model.invoke([fix_prompt])
                fixed_text = fix_result.content.strip()
                detailed_slide = json_to_python(fixed_text)
                detailed_slide = json.loads(is_valid_brackets(detailed_slide))
            except json.JSONDecodeError as e:
                print('last erroe',e)
                print('detailed_slide',detailed_slide)
                detailed_slide = {
                    'slide_number': current_slide['slide_number'],
                    'slide_title': current_slide['slide_title'],
                    'detailed_content':[],
                    'row_response': detailed_slide
                }
    
    detailed_slides = state.get('detailed_slides',[])
    detailed_slides.append(detailed_slide)
    return {
        'messages':[result],
        'detailed_slides':detailed_slides,
        'current_slide_index': current_index +1
    }

def should_coutine_slides(state: PptState):
    """Check if we need to generate more slides"""
    current_index = state.get('current_slide_index',0)
    total_slides = len(state.get('outline',{}).get('slides',[]))

    if current_index < total_slides:
        return "continue"
    else:
        return "end"

def human_decision(state: PptState):
    pass
    
tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node("generate_slide_detail",generate_slide_detail_node)
workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
workflow.add_edge("generate_outline","generate_slide_detail")
workflow.add_conditional_edges(
    "generate_slide_detail",should_coutine_slides,
    {
        "continue":"generate_slide_detail",
        "end":END
    }

)

conn = sqlite3.connect("graph.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

ppt_generator = workflow.compile(checkpointer = checkpointer)


# initial_state = {
#         'messages': [],
#         'outline': {},
#         'detailed_slides': [],
#         'current_slide_index': 0,
#         'topic': 'What is photosynthesis in plants',
#         'num_slides': 1
#     }
# config = {"configurable":{"thread_id":"user-123"}}
# result = ppt_generator.invoke(initial_state,config=config)

# HITL IN OUTLINE

In [9]:
def remove_slides(num_str,slides):
    num = list(map(int,num_str.split(',')))
    print('ritu',num)
    counter = 1
    for i in num:
        print('ritu',i )
        print('ritu',i-counter )
        slides.pop(i-counter)
        counter+=1
    

In [ ]:
OUTLINE_SYSTEM_PORMPT = SystemMessage(
    content="""
You are an expert presentation designer and content strategist.

Your task: Create a STRUCTURED OUTLINE for a PowerPoint presentation.

Output format (JSON):
{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        { 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }
    ]
}

Rules:
- Create a logical flow from introduction to conclusion
- Each slide should have 3-5 key points
- Be specific about what each slide will cover
- Ensure comprehensive coverage of the topic
- Use tools to research if needed for accuracy
- Return ONLY valid JSON, no additional text
"""
)


def except_outline_prompt(outline_text):
    prompt = HumanMessage(
            content=f"""The following text should be valid JSON but has formatting issues:

{outline_text}

Please convert this into valid JSON with the following structure:
{{
    "title":"Main presentation title",
    "totle_slides": number,
    "slides":[
        {{ 
            "slide_number":1,
            "slide_title": "Title of the slide"
            "key_points": ["Point 1","Point 2","Point 3"]
            "content_type": "introduction/explanation/comparison/conclusion"
        }}
    ]
}}

Return ONLY the valid JSON, no explanations or markdown formatting."""
        )
    return prompt 

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    topic: str
    feedback: str
    action: Literal['continue_slide',"continue_next",'update_outline','update_slide']
    num_slides: int
    add_slide: int
    remove_slide: str


def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""

    print('inside generate_outline_node')
    print("state",state)
    
    if 'action' in state and state['action'] == 'update_outline':
        print('inside 11')
        if state['feedback'] =='remove_slide':
            feedback = HumanMessage(content=f'Remove slide number {state["remove_slide"]} from the outline')
            return {
                'messages':[AIMessage(content = json.dumps(state['outline'], indent=2)),feedback],
                'outline':state['outline'],
                'current_slide_index':0
                    }        
        # elif state['feedback'] == 'add_slide':
        #     # add add_slide to the outlines
        #     pass
    
        # else:
        #     if "add_slide" in state and state['add_slide'] > 0:
        #         # update outline acording to the feesback  and slide
        #         pass
        #     else:
        #         # update outline acording to the feesback
        #         pass
    


    # topic = state["topic"]
    # num_slides = state['num_slides']

    # prompt = HumanMessage(
    #     content=f"Create a {num_slides}-slide presentation outline on: {topic}"
    # )

    # messages = [OUTLINE_SYSTEM_PORMPT,prompt]


    messages = state["messages"] + [OUTLINE_SYSTEM_PORMPT]

    result = model_with_tools.invoke(messages)


    print('result',result)
    
    output = {
        'messages':[result],
        'current_slide_index':0
            } 
    if result.content:
        try:
            outline = json_to_python(result.content)
            print('outline 1',outline)
            outline = json.loads(outline)
            print('outline 2',outline)
        except json.JSONDecodeError:
            outline = json.loads(is_valid_brackets(outline))
            print('outline 3',outline)
        output['outline'] = outline
        
    print("state before ",state)
    return output

def generate_slide_detail_node(state: PptState):
    return {
        'messages':[AIMessage('this is generate_slide_detail_node')],
        'detailed_slides':{'ritu':'Ritu this is generate_slide_detail_node'},
        'current_slide_index': 1
    }

def human_decision(state: PptState):
    decision = interrupt({})
    print('inside human_decision')
    print('decision',decision)
    print('state',state)

    if decision['action'] == "update_outline":
        print('inside first 1')

        if 'feedback' in  decision:
            print('inside first 2')
            if 'add_slide' in decision:
                print('inside first 3')
                # need to improve
                # return {
                #     'action': "update_outline",
                #     "feedback":decision['feedback'],
                #     "num_slides":int(decision['add_slide'])+state['num_slides'],
                #     "add_slide":int(decision['add_slide'])
                #     }
            elif 'remove_slide' in decision:
                print('inside first 4')
                # # need to improve
                # remove_slides(decision['remove_slide'],state['outline'])
                # # state['outline']['totle_slides'] -= len(decision['remove_slide'].split(','))
                # return {
                #     'action': "update_outline",
                #     "remove_slide":decision['remove_slide'],
                #     "feedback":decision['feedback'],
                #     "num_slides":state['num_slides']- len(decision['remove_slide'].split(','))
                #     }
            else:
                return {
                    'action': "update_outline",
                    "feedback":decision['feedback'],
                    "messages":[decision['feedback']]
                    }
        elif 'add_slide' in decision:
            print('inside first 5')
            return {
                'action': "update_outline",
                "feedback":'add_slide',
                "num_slides":int(decision['add_slide'])+state['num_slides'],
                "add_slide":int(decision['add_slide'])
                }
        elif 'remove_slide' in decision:
            print('inside first 6')
            # remove_slides(decision['remove_slide'],state['outline']['slides'])
            remove_slides(decision['remove_slide'],state['outline'])
            # state['outline']['totle_slides'] -= len(decision['remove_slide'].split(','))
            print('state',state)
            return {
                'action': "update_outline",
                "remove_slide":decision['remove_slide'],
                "feedback":'remove_slide',
                "num_slides":state['num_slides']- len(decision['remove_slide'].split(','))
                }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}

    
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "update_outline"

    elif action in ('continue_slide', 'continue_next', 'update_slide'):
        return "continue_slide"

DB_URL = os.getenv("ppt_url")

tool_node = ToolNode(tools)

workflow = StateGraph(PptState)
workflow.add_node('generate_outline',generate_outline_node)
workflow.add_node('human_decision',human_decision)
workflow.add_node('generate_slide_detail',generate_slide_detail_node)
# workflow.add_node('chat_node',chat_node)
workflow.add_node('tools',tool_node)

workflow.add_edge(START,'generate_outline')
# workflow.add_conditional_edges('generate_outline',tools_condition)
workflow.add_conditional_edges(
    "generate_outline",
    tools_condition,
    {
        "tools": "tools",
        "__end__": "human_decision",
    },
)

workflow.add_edge("tools", "generate_outline")
workflow.add_conditional_edges('human_decision',
    route_after_human,{
    "update_outline":"generate_outline",
    "continue_slide":"generate_slide_detail",
})
workflow.add_edge('generate_slide_detail',END)


from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool

connection_kwargs = {
        "autocommit": True,
        "prepare_threshold": 0,
    }

pool = ConnectionPool(
       conninfo=DB_URL,
        max_size=20,
        kwargs=connection_kwargs,
)

checkpointer = PostgresSaver(pool)
checkpointer.setup()
graph = workflow.compile(checkpointer=checkpointer)


SyntaxError: invalid syntax (3316513924.py, line 153)

In [25]:
# content=f"Create a {num_slides}-slide presentation outline on: {topic}
num_slides = 3
topic = 'resursion in python'
result = graph.invoke({"messages":HumanMessage(content=f"Create a {num_slides}-slide presentation outline on: {topic}"),
                      "num_slides":num_slides},
                      config = {"configurable":{"thread_id":116}}
)

inside generate_outline_node
state {'messages': [HumanMessage(content='Create a 3-slide presentation outline on: resursion in python', additional_kwargs={}, response_metadata={}, id='8624f2b5-d790-40a3-a91b-059751a53f2a')], 'num_slides': 3}
result content='' additional_kwargs={} response_metadata={} id='lc_run--019bf66b-bf4a-7792-a935-8b902af7f8e8-0' tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'recursion in python'}, 'id': 'chatcmpl-tool-9c36f8ab47895308', 'type': 'tool_call'}] invalid_tool_calls=[]
state before  {'messages': [HumanMessage(content='Create a 3-slide presentation outline on: resursion in python', additional_kwargs={}, response_metadata={}, id='8624f2b5-d790-40a3-a91b-059751a53f2a')], 'num_slides': 3}
inside generate_outline_node
state {'messages': [HumanMessage(content='Create a 3-slide presentation outline on: resursion in python', additional_kwargs={}, response_metadata={}, id='8624f2b5-d790-40a3-a91b-059751a53f2a'), AIMessage(content='', addit

In [ ]:
# input_data = Command(
#     resume={
#         "action":'update_outline',
#         "feedback":'can you also add images refrence'
        
#     }
# )
# result1 = graph.invoke(input_data,
#     config = {"configurable":{"thread_id":116}})